# Bronze Ingestion
This notebook implements Bronze-layer ingestion from ADLS Gen2 using Databricks Auto Loader.
Auto Loader is used to support incremental file ingestion and checkpoint-based processing, which makes the notebook suitable for repeated execution as a Databricks Job.

In [0]:

from pyspark.sql import functions as F

# Parameters supplied by the Databricks Job / Bundle
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "lab8_dev_bronze")
dbutils.widgets.text("raw_volume", "raw_files")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_volume = dbutils.widgets.get("raw_volume")

# Input CSV files in the current environment
base_path = f"/Volumes/{catalog}/{bronze_schema}/{raw_volume}"
state_path = f"/Volumes/{catalog}/{bronze_schema}/autoloader_state"

climatic_raw_path = base_path

# Auto Loader state: separate paths for each source
climatic_checkpoint_path = f"{state_path}/checkpoints/yield_climate"
climatic_schema_path = f"{state_path}/schemas/yield_climate"

# Target Bronze Delta table
target_table = f"{catalog}.{bronze_schema}.yield_and_climate"


### Read source data

In [0]:

df_climatic = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", climatic_schema_path)
    .option("pathGlobFilter", "europe_cereal_yield_climate_1990_2022.csv")
    .load(climatic_raw_path)
)

### Add ingestion metadata

In [0]:
df_climatic_bronze = (
    df_climatic
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

### Write to the Bronze layer

In [0]:
query = (
    df_climatic_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", climatic_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

## Repeating the same loop for another table

In [0]:

environment_raw_path = base_path

environment_checkpoint_path = f"{state_path}/checkpoints/environment"
environment_schema_path = f"{state_path}/schemas/environment"

environment_target_table = f"{catalog}.{bronze_schema}.environment"



### Read source data

In [0]:
df_environment = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", environment_schema_path)
    .option("pathGlobFilter", "environment_clean.csv")
    .load(environment_raw_path)
)

### Add ingestion metadata

In [0]:
df_environment_bronze = (
    df_environment
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

### Write to the Bronze layer

In [0]:
environment_query = (
    df_environment_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", environment_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(environment_target_table)
)

environment_query.awaitTermination()